# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

# ---------------------------------------------------------------
# Mapa leído de la imagen:
#
#        c0    c1    c2    c3    c4    c5
#  r0    S     .     .     #     .   +10
#  r1    .     #     ~     .    -3     .
#  r2    .     ~    +2     .     #     .
#  r3    .     .     .     ~     .   -10
#  r4    .    -3     #     .     .     .
#
#   #  = estantería / pared          ~  = piso resbaloso
#  +10 = entrega (terminal)         +2 = carga (terminal)
#  -10 = peligro mortal (terminal)  -3 = peligro (NO terminal)
# ---------------------------------------------------------------

class WarehouseMDP:
    """
    MDP del almacén, con la notación de clase:
        state  = (row, col)
        action = (dr, dc)
        R(s)   = recompensa del estado actual
        T(s,a,s') = P(s' | s,a)   -> depende del tipo de piso de s
    """

    def __init__(self, living_reward=-1.0, gamma=0.9,
                 p_slippery_intended=0.60):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterías / paredes: no son estados.
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso: cambia la dinámica cuando el robot ESTÁ aquí.
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Terminales: (row, col) -> R(s)
        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2):  +2.0,   # estación de carga
            (3, 5): -10.0,   # peligro mortal
        }

        # Peligros no terminales: se puede pasar por ellos y seguir.
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Dinámica: piso normal vs piso resbaloso
        self.p_normal_intended = 0.90
        self.p_normal_deviation = (1.0 - self.p_normal_intended) / 2   # 0.05

        self.p_slippery_intended = p_slippery_intended
        self.p_slippery_deviation = (1.0 - self.p_slippery_intended) / 2

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): sólo depende del estado en el que está el robot.
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...] para T(s,a,s').
        """
        # Los terminales son absorbentes.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # La ESTOCASTICIDAD depende del piso del estado actual s,
        # no de la acción ni del estado destino.
        if state in self.slippery_states:
            p_intended = self.p_slippery_intended
            p_deviation = self.p_slippery_deviation
        else:
            p_intended = self.p_normal_intended
            p_deviation = self.p_normal_deviation

        # Desviaciones perpendiculares a (dr, dc).
        perp1 = ( action[1],  action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1,  p_deviation),
            (perp2,  p_deviation),
        ]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])

            # Fuera del grid o contra una estantería -> se queda quieto.
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    # Σ_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Actualización sincrónica: V_{k+1} se calcula sólo con V_k.
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # max_a Σ_{s'} T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                # V_{k+1}(s) = R(s) + gamma * max_a Σ T V_k
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state]),
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a Σ_{s'} T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V),
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """V^pi(s) = R(s) + gamma * Σ_{s'} T(s,pi(s),s') V^pi(s')  (sin max)"""
    V = {state: 0.0 for state in grid.states()}

    for _ in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(
                        grid, state, policy[state], V
                    )
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state]),
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V


def policy_improvement(grid, V):
    # Mismo argmax que extract_policy: mejora voraz respecto a V^pi.
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: todos hacia arriba.
    policy = {
        state: (-1, 0)
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []   # cuántos estados cambiaron de acción en cada iteración

    for _ in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changes = sum(
            1 for state in policy if new_policy[state] != policy[state]
        )
        history.append(changes)
        policy = new_policy

        # 4. estabilidad
        if changes == 0:
            break

    return policy, V, history


## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [ ]:
def resolver(titulo, **kwargs):
    g = WarehouseMDP(**kwargs)
    V, n = value_iteration(g)
    pi = extract_policy(g, V)
    print(f"=== {titulo} ===  (iteraciones: {n})")
    print_policy(g, pi)
    print(f"V(START) = {V[g.start]:+.3f}\n")
    return g, V, pi


resolver("BASE            living=-1.0, gamma=0.9, slip=0.60")
resolver("A  living=-0.1", living_reward=-0.1)
resolver("B  slip=0.40",   p_slippery_intended=0.40)
resolver("C  gamma=0.99",  gamma=0.99)

In [ ]:
# --- BONUS: ¿en qué living_reward cambia el destino desde START? ---

def destino_desde_start(grid, policy):
    """Sigue la política (ignorando el ruido) y devuelve el terminal alcanzado."""
    s = grid.start
    visitados = set()
    while not grid.is_terminal(s) and s not in visitados:
        visitados.add(s)
        a = policy[s]
        siguiente = (s[0] + a[0], s[1] + a[1])
        s = siguiente if grid.is_valid_state(siguiente) else s
    return s if grid.is_terminal(s) else None


def barrido(nombre, valores, construir):
    print(f"== {nombre} ==")
    anterior = None
    for v in valores:
        g = construir(v)
        V, _ = value_iteration(g)
        d = destino_desde_start(g, extract_policy(g, V))
        marca = "  <-- CAMBIO" if anterior is not None and d != anterior else ""
        print(f"  {v:+.2f}  destino {d}   V(START)={V[g.start]:+.3f}{marca}")
        anterior = d
    print()


# Umbral de living_reward (afinado de 0.01 en 0.01 alrededor del cambio)
barrido(
    "living_reward",
    [round(-0.70 - 0.01 * i, 2) for i in range(15)],
    lambda v: WarehouseMDP(living_reward=v),
)

# Umbral de gamma
barrido(
    "gamma",
    [0.90, 0.92, 0.94, 0.96, 0.99],
    lambda v: WarehouseMDP(gamma=v),
)

## Respuestas — interpretación de la política

El mapa (26 estados, 4 estanterías) es:

|  | c0 | c1 | c2 | c3 | c4 | c5 |
|---|---|---|---|---|---|---|
| **r0** | START | . | . | ▉ | . | **+10** |
| **r1** | . | ▉ | ~ | . | −3 | . |
| **r2** | . | ~ | **+2** | . | ▉ | . |
| **r3** | . | . | . | ~ | . | **−10** |
| **r4** | . | −3 | ▉ | . | . | . |

Política óptima (base: `living=-1`, `γ=0.9`, slip `0.60`):

```
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑
 →  |  →  | +2 |  ↑  |  #  |  ↑
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ←
```

**1. Desde `START`, ¿entrega +10 o carga +2?**
Va a la **carga `+2`**. La trayectoria es `(0,0) → (0,1) → (0,2) ↓ (1,2) ↓ (2,2)`:
tres pasos y termina. $V(\text{START}) = -2.575$, que es menos que el `+2` porque
paga ~3 pasos de `-1` en el camino.

La decisión se toma en **`(1,2)`**, que es piso resbaloso. Desde ahí el robot puede
bajar al `+2` (1 paso) o girar a la derecha por el pasillo `(1,3) → (1,4) → (1,5) → (0,5)`
(4 pasos más), pasando obligatoriamente por el peligro `-3` de `(1,4)`. La política elige `↓`.

**2. ¿Por qué una recompensa menor puede ser óptima?**
Porque lo que se maximiza es el **retorno descontado esperado**, no la recompensa final.
El camino al `+10` cuesta 4 pasos extra de `-1`, el `-3` de `(1,4)`, y llega descontado:
$0.9^4 \cdot 10 \approx 6.6$. Restando el `-3` y los pasos, queda por debajo del `+2`
que está a un solo paso. El `+2` gana **por cercanía y por seguridad**, no por magnitud.

Observa además el gradiente de valores: `(0,4)` vale `+7.607` y `(1,5)` también `+7.607`.
Cerca de la entrega, el `+10` sí domina — el problema es sólo llegar hasta allá desde `START`.

**3. ¿Dónde el piso resbaloso cambia la decisión?**
Los tres estados resbalosos son `(1,2)`, `(2,1)` y `(3,3)`:

- **`(1,2)`** es el estado crítico. Es justo donde se bifurca ir al `+2` o al `+10`,
  y es resbaloso: con `p=0.6`, apuntar a la derecha (`→`) tiene `0.2` de acabar en
  `(0,2)` y `0.2` en `(2,2)`. Ese ruido encarece el camino largo y refuerza el `↓` al `+2`.
- **`(3,3)`** está en el corredor inferior, cerca del `-10` de `(3,5)`. La política ahí
  es `↑` (subir hacia `(2,3)`), no avanzar por la fila 3.
- Toda la mitad inferior del grid tiene valores negativos (`(4,1) = -3.890`), así que la
  política empuja hacia arriba: el robot nunca quiere estar ahí.

**4. ¿Qué papel cumple el `-1` por paso?**
Es la **presión de urgencia** que hace que el `+2` cercano gane al `+10` lejano. Es
exactamente el parámetro que decide la respuesta a la pregunta 1: con `-1` el robot se
conforma; con `-0.1` cruza medio almacén por el `+10` (experimento A).

**5. ¿Por qué $T(s,a,s')$ ya no puede usar las mismas probabilidades en todos los estados?**
Porque el ruido es una propiedad **del estado actual** (el tipo de piso), no del modelo
global. `get_transition_probs` consulta `state in self.slippery_states` y elige
`(0.9, 0.05, 0.05)` o `(0.6, 0.2, 0.2)` antes de construir la distribución. El MDP sigue
siendo estacionario —las probabilidades no cambian con el tiempo—, pero **no** es homogéneo
en el espacio de estados.

---

## Resultados de los experimentos

| | destino desde `START` | $V(\text{START})$ | iteraciones VI |
|---|---|---|---|
| **Base** (`-1`, `γ=0.9`, `0.60`) | carga **+2** `(2,2)` | `-2.575` | 20 |
| **A** `living_reward=-0.1` | entrega **+10** `(0,5)` | `+1.649` | 26 |
| **B** slip `0.60 → 0.40` | carga **+2** (política idéntica a la base) | `-2.706` | 22 |
| **C** `γ=0.99` | entrega **+10** `(0,5)` | `-1.456` | 24 |

**Experimento A — menos costo por paso.**
La política **cambia** en `(1,2)`: de `↓` (al `+2`) pasa a `→`. Con pasos casi gratis el
rodeo por `(1,3) → (1,4) → (1,5) → (0,5)` deja de ser caro y el `-3` de `(1,4)` se vuelve
un peaje aceptable a cambio del `+10`. También cambia `(1,0)` de `↓` a `↑` y `(3,2)` de
`↑` a `→`. Es el resultado esperado: el `-1` era lo único que sostenía la elección del `+2`.

**Experimento B — piso muy resbaloso (0.40).**
La política óptima es **idéntica** a la base; sólo bajan los valores (`V(START)` de
`-2.575` a `-2.706`). Es un resultado instructivo: la política base ya evitaba los
corredores riesgosos, así que hacer el piso *más* resbaloso no le da nada nuevo que
evitar — sólo lo hace más costoso pasar por `(1,2)` y `(2,1)`, que ya eran obligatorios.
**Más ruido no siempre cambia la política; a veces sólo empeora el valor.**

**Experimento C — más paciencia (`γ=0.99`).**
Aquí **sí** cambia: `(1,2)` pasa a `→` y el robot va por el `+10`. Con $\gamma=0.99$
el `+10` a 4 pasos vale $0.99^4 \cdot 10 \approx 9.6$ en lugar de $6.6$, y eso alcanza
para pagar el `-3` y los pasos extra. El umbral está entre $\gamma = 0.92$ y $\gamma = 0.94$:

| γ | 0.90 | 0.92 | 0.94 | 0.96 | 0.99 |
|---|---|---|---|---|---|
| destino | +2 | +2 | **+10** | +10 | +10 |
| V(START) | −2.575 | −2.525 | −2.339 | −2.050 | −1.456 |

**Bonus — umbral de `living_reward`.**
El cambio de destino ocurre entre **`-0.79` y `-0.80`**:

- `living_reward ≳ -0.79` → intenta la **entrega +10**;
- `living_reward ≲ -0.80` → se conforma con la **carga +2**.

Nota que el umbral es bastante alto (el robot abandona el `+10` con un costo por paso
menor a 1). La razón es que el camino al `+10` no sólo es más largo: obliga a pasar por
el `-3` de `(1,4)`, así que el costo efectivo del desvío es ~4 pasos **más** 3 unidades.